# Kaede Axon Local Translation Analysis: Tagged vs Untagged

This notebook analyzes processed green/red Kaede ratio traces along actively extending RGC axons.

## Input data assumed

Two folders of CSV files:

```text
kaede_axon_project/
    data/
        tagged/
            axon_001.csv
            axon_002.csv
        untagged/
            axon_001.csv
            axon_002.csv
```

Each CSV represents one axon.

CSV structure:

```text
rows    = pixels along axon trace
columns = timepoints
values  = green/red fluorescence ratio
```

Orientation:

```text
row 0    = distal growth cone / leading tip
last row = proximal axon near optic chiasm
```

Timepoints:

```text
t=-1, t=0, t=1, ..., t=9
```

where t=0 to t=9 are 0 to 90 minutes after photoconversion, sampled every 10 minutes.

## Main hypothesis

Tagged Kaede should show faster green/red recovery near the distal growth cone than untagged Kaede.

## Required libraries and purpose

| Library | Purpose |
|---|---|
| `pathlib` | file/folder paths |
| `numpy` | numerical calculations and slope fitting |
| `pandas` | load CSV files and organize tables |
| `matplotlib` | plotting |
| `seaborn` | group plots and heatmaps |
| `scipy.stats` | group statistical tests |
| `statsmodels` | regression and mixed-effects models |

Install:

```bash
pip install numpy pandas matplotlib seaborn scipy statsmodels
```

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 40)

sns.set_context("notebook")
sns.set_style("whitegrid")

## Step 1 — Set paths

Update `project_dir` to your actual project folder.

In [ ]:
project_dir = Path("../kaede_axon_project")

tagged_dir = project_dir / "data" / "tagged"
untagged_dir = project_dir / "data" / "untagged"

output_dir = project_dir / "outputs"
table_dir = output_dir / "tables"
fig_dir = output_dir / "figures"

for folder in [output_dir, table_dir, fig_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print("Tagged folder:", tagged_dir)
print("Untagged folder:", untagged_dir)

## Step 2 — Helper functions

These functions:

1. Load one axon CSV.
2. Convert pixel rows into normalized axon position.
3. Convert the matrix into long-format data.
4. Compute slope of green/red ratio over time.

In [ ]:
def load_ratio_csv(csv_path):
    csv_path = Path(csv_path)
    df = pd.read_csv(csv_path)

    # Drop first column if it appears to be a pixel/position/index column.
    first_col = df.columns[0]
    first_numeric = pd.to_numeric(df[first_col], errors="coerce")

    if first_numeric.isna().any() or str(first_col).lower() in ["pixel", "position", "distance", "index"]:
        df = df.drop(columns=[first_col])

    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.dropna(axis=0, how="all")
    df = df.dropna(axis=1, how="all")

    return df


def make_time_values(n_timepoints, minutes_per_step=10):
    # First column is t=-1, then t=0 to t=9.
    return np.arange(-1, n_timepoints - 1) * minutes_per_step


def matrix_to_long(matrix_df, axon_id, group, minutes_per_step=10):
    n_pixels = matrix_df.shape[0]
    n_timepoints = matrix_df.shape[1]

    row_index = np.arange(n_pixels)

    # Original row 0 is distal. Convert so 0 = proximal, 100 = distal.
    distal_to_proximal = 100 * row_index / (n_pixels - 1)
    position_percent = 100 - distal_to_proximal

    time_minutes = make_time_values(n_timepoints, minutes_per_step)

    rows = []

    for pix_idx in range(n_pixels):
        for time_idx in range(n_timepoints):
            rows.append({
                "axon_id": axon_id,
                "group": group,
                "pixel_index_original": pix_idx,
                "position_percent": position_percent[pix_idx],
                "time_index": time_idx,
                "time_min": time_minutes[time_idx],
                "ratio_green_red": matrix_df.iloc[pix_idx, time_idx]
            })

    return pd.DataFrame(rows)


def assign_position_bin(position_percent, bin_size=10):
    if pd.isna(position_percent):
        return np.nan

    bin_start = int(np.floor(position_percent / bin_size) * bin_size)

    if bin_start >= 100:
        bin_start = 100 - bin_size

    bin_end = bin_start + bin_size

    return f"{bin_start:03d}_{bin_end:03d}"


def slope_from_timecourse(df, y_col="ratio_green_red"):
    clean = df[["time_min", y_col]].dropna()

    if clean.shape[0] < 3:
        return np.nan

    x = clean["time_min"].values
    y = clean[y_col].values

    slope, intercept = np.polyfit(x, y, 1)

    return slope

## Step 3 — Load all axon CSV files

Output:

```text
kaede_ratio_long_format.csv
```

This table contains one row per axon-position-timepoint.

In [ ]:
def load_group_folder(folder, group_name):
    folder = Path(folder)
    csv_files = sorted(folder.glob("*.csv"))

    long_tables = []

    for csv_path in csv_files:
        axon_id = csv_path.stem
        matrix_df = load_ratio_csv(csv_path)

        long_df = matrix_to_long(
            matrix_df,
            axon_id=axon_id,
            group=group_name,
            minutes_per_step=10
        )

        long_tables.append(long_df)

    if len(long_tables) == 0:
        return pd.DataFrame()

    return pd.concat(long_tables, ignore_index=True)


tagged_long = load_group_folder(tagged_dir, "tagged")
untagged_long = load_group_folder(untagged_dir, "untagged")

long_df = pd.concat([tagged_long, untagged_long], ignore_index=True)

print("Long table shape:", long_df.shape)
display(long_df.head())

long_path = table_dir / "kaede_ratio_long_format.csv"
long_df.to_csv(long_path, index=False)

print("Saved:", long_path)

## Step 4 — Quality control

Checks number of axons, pixels, timepoints, and missing values.

In [ ]:
qc_summary = (
    long_df
    .groupby(["group", "axon_id"])
    .agg(
        n_rows=("ratio_green_red", "size"),
        n_valid_values=("ratio_green_red", "count"),
        n_pixels=("pixel_index_original", "nunique"),
        n_timepoints=("time_min", "nunique"),
        min_position=("position_percent", "min"),
        max_position=("position_percent", "max"),
        min_time=("time_min", "min"),
        max_time=("time_min", "max")
    )
    .reset_index()
)

qc_path = table_dir / "axon_qc_summary.csv"
qc_summary.to_csv(qc_path, index=False)

display(qc_summary)
print("Saved:", qc_path)

## Step 5 — Bin axon position

Default bins:

```text
0-10%, 10-20%, ..., 90-100%
```

where:

```text
0% = proximal
100% = distal growth cone
```

In [ ]:
BIN_SIZE = 10

long_df["position_bin"] = long_df["position_percent"].apply(
    lambda x: assign_position_bin(x, bin_size=BIN_SIZE)
)

long_df["position_bin_start"] = long_df["position_bin"].str.split("_").str[0].astype(float)
long_df["position_bin_end"] = long_df["position_bin"].str.split("_").str[1].astype(float)
long_df["position_bin_center"] = (
    long_df["position_bin_start"] + long_df["position_bin_end"]
) / 2

display(long_df.head())

## Step 6 — Plot ratio over time by axon position

This shows whether green/red ratio increases preferentially near the distal growth cone.

In [ ]:
mean_time_position = (
    long_df
    .groupby(["group", "position_bin_center", "time_min"])
    .agg(
        mean_ratio=("ratio_green_red", "mean"),
        sem_ratio=("ratio_green_red", lambda x: x.std() / np.sqrt(x.count()))
    )
    .reset_index()
)

for group_name in mean_time_position["group"].unique():

    plot_df = mean_time_position[mean_time_position["group"] == group_name]

    plt.figure(figsize=(9, 6))

    for pos in sorted(plot_df["position_bin_center"].unique()):
        sub = plot_df[plot_df["position_bin_center"] == pos]
        plt.plot(sub["time_min"], sub["mean_ratio"], marker="o", label=f"{pos:.0f}%")

    plt.xlabel("Time after photoconversion (min)")
    plt.ylabel("Mean green/red ratio")
    plt.title(f"Green/Red ratio over time by position: {group_name}")
    plt.legend(title="Position", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()

    out_path = fig_dir / f"{group_name}_ratio_over_time_by_position.tif"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", out_path)

## Step 7 — Ratio kymograph by group

Kymograph:

```text
x-axis = time
y-axis = axon position
color = green/red ratio
```

In [ ]:
for group_name in long_df["group"].unique():

    plot_df = (
        long_df[long_df["group"] == group_name]
        .groupby(["position_bin_center", "time_min"])
        .agg(mean_ratio=("ratio_green_red", "mean"))
        .reset_index()
    )

    heatmap_df = plot_df.pivot(
        index="position_bin_center",
        columns="time_min",
        values="mean_ratio"
    ).sort_index(ascending=False)

    plt.figure(figsize=(9, 6))

    sns.heatmap(
        heatmap_df,
        cmap="viridis",
        cbar_kws={"label": "Mean green/red ratio"}
    )

    plt.xlabel("Time after photoconversion (min)")
    plt.ylabel("Axon position (%; distal at top)")
    plt.title(f"Green/Red ratio kymograph: {group_name}")
    plt.tight_layout()

    out_path = fig_dir / f"{group_name}_ratio_kymograph.tif"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", out_path)

## Step 8 — Compute slope of ratio increase over time

For each axon and position bin:

```text
green/red ratio ~ time
```

Slope = rate of ratio increase per minute.

In [ ]:
slope_rows = []

for (group_name, axon_id, position_bin), sub in long_df.groupby(
    ["group", "axon_id", "position_bin"]
):
    slope = slope_from_timecourse(sub)
    pos_center = sub["position_bin_center"].iloc[0]

    slope_rows.append({
        "group": group_name,
        "axon_id": axon_id,
        "position_bin": position_bin,
        "position_bin_center": pos_center,
        "ratio_slope_per_min": slope
    })

slope_df = pd.DataFrame(slope_rows)

slope_path = table_dir / "kaede_ratio_slope_by_axon_position.csv"
slope_df.to_csv(slope_path, index=False)

display(slope_df.head())
print("Saved:", slope_path)

## Step 9 — Plot slope vs axon position

This is the key spatial profile.

Expected tagged pattern:

```text
slope increases toward 100% distal growth cone
```

In [ ]:
plt.figure(figsize=(8, 5))

sns.lineplot(
    data=slope_df,
    x="position_bin_center",
    y="ratio_slope_per_min",
    hue="group",
    marker="o",
    errorbar="se"
)

plt.axhline(0, linestyle="--", color="black")

plt.xlabel("Axon position (%; 0=proximal, 100=distal growth cone)")
plt.ylabel("Slope of green/red ratio increase per min")
plt.title("Spatial profile of Kaede green/red recovery")
plt.tight_layout()

out_path = fig_dir / "slope_vs_axon_position_by_group.tif"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out_path)

## Step 10 — Distal enrichment metric

Define:

```text
proximal = 0-30%
distal   = 70-100%
```

Primary metric:

```text
distal_minus_proximal_slope = distal_slope - proximal_slope
```

This gives one value per axon.

In [ ]:
PROXIMAL_MAX = 30
DISTAL_MIN = 70

axon_metric_rows = []

for (group_name, axon_id), sub in slope_df.groupby(["group", "axon_id"]):

    proximal = sub[sub["position_bin_center"] <= PROXIMAL_MAX]["ratio_slope_per_min"]
    distal = sub[sub["position_bin_center"] >= DISTAL_MIN]["ratio_slope_per_min"]

    proximal_mean = proximal.mean()
    distal_mean = distal.mean()

    axon_metric_rows.append({
        "group": group_name,
        "axon_id": axon_id,
        "proximal_slope_mean": proximal_mean,
        "distal_slope_mean": distal_mean,
        "distal_minus_proximal_slope": distal_mean - proximal_mean,
        "distal_over_proximal_slope": np.nan if np.isclose(proximal_mean, 0) or pd.isna(proximal_mean) else distal_mean / proximal_mean
    })

axon_metrics = pd.DataFrame(axon_metric_rows)

metric_path = table_dir / "kaede_distal_enrichment_metrics_by_axon.csv"
axon_metrics.to_csv(metric_path, index=False)

display(axon_metrics)
print("Saved:", metric_path)

## Step 11 — Compare tagged vs untagged

Primary test:

```text
tagged distal_minus_proximal_slope
vs
untagged distal_minus_proximal_slope
```

In [ ]:
plt.figure(figsize=(7, 5))

sns.boxplot(
    data=axon_metrics,
    x="group",
    y="distal_minus_proximal_slope"
)

sns.stripplot(
    data=axon_metrics,
    x="group",
    y="distal_minus_proximal_slope",
    color="black",
    alpha=0.7
)

plt.axhline(0, linestyle="--", color="black")

plt.xlabel("Group")
plt.ylabel("Distal - proximal slope")
plt.title("Distal enrichment of green/red recovery")
plt.tight_layout()

out_path = fig_dir / "distal_minus_proximal_slope_by_group.tif"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out_path)

In [ ]:
tagged_values = axon_metrics.loc[
    axon_metrics["group"] == "tagged",
    "distal_minus_proximal_slope"
].dropna()

untagged_values = axon_metrics.loc[
    axon_metrics["group"] == "untagged",
    "distal_minus_proximal_slope"
].dropna()

stat_rows = []

if len(tagged_values) > 1 and len(untagged_values) > 1:

    t_stat, t_p = stats.ttest_ind(tagged_values, untagged_values, equal_var=False)
    u_stat, u_p = stats.mannwhitneyu(tagged_values, untagged_values, alternative="two-sided")

    stat_rows.append({
        "metric": "distal_minus_proximal_slope",
        "test": "Welch_t_test",
        "statistic": t_stat,
        "p_value": t_p
    })

    stat_rows.append({
        "metric": "distal_minus_proximal_slope",
        "test": "Mann_Whitney_U",
        "statistic": u_stat,
        "p_value": u_p
    })

stats_df = pd.DataFrame(stat_rows)

stats_path = table_dir / "kaede_group_comparison_statistics.csv"
stats_df.to_csv(stats_path, index=False)

display(stats_df)
print("Saved:", stats_path)

## Step 12 — Regression: slope as a function of position and group

This model tests:

```text
slope ~ group * position
```

The interaction term tests whether the tagged group has a different spatial slope profile along the axon.

In [ ]:
regression_df = slope_df.dropna(
    subset=["ratio_slope_per_min", "position_bin_center", "group"]
).copy()

regression_df["position_scaled"] = regression_df["position_bin_center"] / 100

try:
    model = smf.ols(
        "ratio_slope_per_min ~ C(group) * position_scaled",
        data=regression_df
    ).fit()

    print(model.summary())

    model_path = table_dir / "slope_position_group_regression_summary.txt"
    with open(model_path, "w") as f:
        f.write(model.summary().as_text())

    print("Saved:", model_path)

except Exception as e:
    print("Regression model could not be fit:", e)

## Step 13 — Mixed-effects model by axon

Because multiple position bins come from each axon, a mixed model is better:

```text
slope ~ group * position + axon_effect
```

In [ ]:
try:
    mixed_model = smf.mixedlm(
        "ratio_slope_per_min ~ C(group) * position_scaled",
        data=regression_df,
        groups=regression_df["axon_id"]
    ).fit()

    print(mixed_model.summary())

    mixed_path = table_dir / "slope_position_group_mixed_model_summary.txt"
    with open(mixed_path, "w") as f:
        f.write(mixed_model.summary().as_text())

    print("Saved:", mixed_path)

except Exception as e:
    print("Mixed model could not be fit:", e)

## Final interpretation

Key outputs:

| Output | Meaning |
|---|---|
| ratio kymograph | green/red recovery over time and axon position |
| slope vs position | spatial map of recovery rate |
| distal slope | recovery near growth cone |
| proximal slope | recovery near proximal axon |
| distal - proximal slope | growth-cone enrichment metric |
| group comparison | tagged vs untagged evidence |

Expected result if the tag promotes growth-cone local translation:

```text
tagged:
    distal slope > proximal slope

untagged:
    distal slope ≈ proximal slope
```

Most important final metric:

```text
distal_minus_proximal_slope
```